# 02 · Tier 0 baselines: logistic regression, XGBoost, MLP

One unit per dataset × seed. These three models see only the tabular features. **XGBoost is the reference
every later tier is compared against**, because gradient-boosted trees on tabular features are what banks
deploy today. Logistic regression is the shape of a traditional scorecard. The MLP tells us whether a neural
network on the same features can keep up (on DGraph-Fin the paper reports MLP 0.723 AUC).

Outputs in the run directory: `predictions/<model>.parquet` (val and test), `models/xgb.json`, `metrics.json`
with a paired-bootstrap comparison of each model against LR, and figures.

**How this notebook runs.** Never by hand in an editor. `scripts/run_nb.sh 02_baselines_tabular dataset=... seed=...` executes it
with papermill on the GPU node, writes the executed copy to `runs/02_baselines_tabular/<dataset>/<graph_builder>/<seed>/executed.ipynb`,
and the last cell writes a `DONE` marker. If the process dies, the wrapper writes `FAILED` and the next `run_stage.sh`
picks the unit up again; checkpoints inside the run directory mean little work is repeated.

In [ ]:
# Parameters (papermill overrides these; keep names and types)
dataset = "taiwan"
seed = 0
graph_builder = "none"
run_dir = ""       # empty -> runs/<stage>/<dataset>/<graph_builder>/<seed>
force = 0          # 1 -> redo work even if checkpoints exist
quick = 0          # 1 -> subsampled data, few epochs; smoke test only

In [ ]:
import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import ckpt, config, paths, viz
from src.eval import results

STAGE = "02_baselines_tabular"
warnings.filterwarnings("ignore")
seed = int(seed)
force, quick = bool(int(force)), bool(int(quick))
run_dir = Path(run_dir) if run_dir else paths.run_dir(STAGE, dataset, graph_builder, seed)
run_dir.mkdir(parents=True, exist_ok=True)
t0 = time.time()
print(f"stage={STAGE} dataset={dataset} graph_builder={graph_builder} seed={seed} quick={quick} force={force}")
print("run_dir:", run_dir)
pd.Series(paths.describe())

## Data

In [ ]:
from src.data import splits
from src.data.base import Bundle

cfg = config.load_config(dataset)
models_cfg = config.quick_override(cfg, quick)
bundle = Bundle.load(paths.processed_dir(dataset, quick))
split = splits.load_split(dataset, quick)
X, y = bundle.X, bundle.y.to_numpy()
tr, va, te = split["train"], split["val"], split["test"]
allowed = np.sort(np.concatenate([tr, va]))          # rows that may take part in graph construction
X_tr, X_va, X_te = X.iloc[tr], X.iloc[va], X.iloc[te]
y_tr, y_va, y_te = y[tr], y[va], y[te]
gdir = paths.graphs_dir(dataset, quick)
n_boot = 200 if quick else cfg["bootstrap"]["n_boot"]
print(bundle.summary())
print({p: len(split[p]) for p in splits.PARTS}, "| test positive rate:", round(float(np.nanmean(y_te)), 4))

## Logistic regression

Median imputation + standardisation for numeric columns, one-hot for categoricals (levels with < 50 rows are pooled).

In [ ]:
from src.eval import metrics
from src.models import lr as LR

preds, timings = {}, {}
t = time.time()
lr_model = LR.fit_lr(X_tr, y_tr, bundle.categorical, C=models_cfg["lr"]["C"], max_iter=models_cfg["lr"]["max_iter"], seed=seed)
preds["lr"] = (LR.predict_lr(lr_model, X_va), LR.predict_lr(lr_model, X_te))
timings["lr"] = round(time.time() - t, 1)
print(f"LR  val AUC {metrics.auc(y_va, preds['lr'][0]):.4f}   test AUC {metrics.auc(y_te, preds['lr'][1]):.4f}   {timings['lr']}s")

## XGBoost

Early stopping on validation AUC. Hyper-parameters come from the dataset config and are the same for every seed; only the seed changes.

In [ ]:
from src.models import xgb as XGB

t = time.time()
xgb_params = dict(models_cfg["xgb"])
booster = XGB.fit_xgb(X_tr, y_tr, X_va, y_va, xgb_params, seed=seed)
preds["xgb"] = (XGB.predict_xgb(booster, X_va), XGB.predict_xgb(booster, X_te))
timings["xgb"] = round(time.time() - t, 1)
XGB.save_xgb(booster, run_dir / "models" / "xgb.json")
print(f"XGB val AUC {metrics.auc(y_va, preds['xgb'][0]):.4f}   test AUC {metrics.auc(y_te, preds['xgb'][1]):.4f}   {timings['xgb']}s")
XGB.summary(booster)

## MLP

Same prepared features as the GNN input later (median impute, standardise, clip). Epoch checkpoints under `ckpt/mlp/`; a killed job resumes at the last saved epoch.

In [ ]:
from src import device

history = []
if device.torch_available():
    from src.models import mlp as MLPM
    from src.models.prep import TabularPrep

    prep = TabularPrep().fit(X_tr)
    Z_tr, Z_va, Z_te = prep.transform(X_tr), prep.transform(X_va), prep.transform(X_te)
    ck = ckpt.TrainingCheckpoint(run_dir, tag="mlp", every=models_cfg["mlp"].get("ckpt_every", 1))
    if force:
        shutil.rmtree(ck.dir, ignore_errors=True)
        ck = ckpt.TrainingCheckpoint(run_dir, tag="mlp", every=models_cfg["mlp"].get("ckpt_every", 1))
    pos_w = float((1 - y_tr.mean()) / y_tr.mean()) if y_tr.mean() < 0.05 else None
    t = time.time()
    mlp_model, history = MLPM.fit_mlp(Z_tr, y_tr, Z_va, y_va, models_cfg["mlp"], ck, seed=seed,
                                      device=device.torch_device(), pos_weight=pos_w)
    preds["mlp"] = (MLPM.predict_mlp(mlp_model, Z_va), MLPM.predict_mlp(mlp_model, Z_te))
    timings["mlp"] = round(time.time() - t, 1)
    print(f"MLP val AUC {metrics.auc(y_va, preds['mlp'][0]):.4f}   test AUC {metrics.auc(y_te, preds['mlp'][1]):.4f}   {timings['mlp']}s")
    fig = viz.plot_training_curves(history, title="MLP")
    viz.savefig(fig, run_dir, "mlp_training")
    plt.show()
else:
    print("torch not installed: MLP skipped (fine on a laptop, not fine on the node)")

## Test metrics

AUC/Gini/KS rank quality, PR-AUC and capture@10% for the rare positive class, Brier and ECE for calibration before any post-hoc calibration.

In [ ]:
rows = {name: results.metrics_block(y_va, pv, y_te, pt) for name, (pv, pt) in preds.items()}
table = pd.DataFrame({m: r["test"] for m, r in rows.items()}).T[
    ["auc", "gini", "ks", "pr_auc", "capture_10", "bad_rate_top10", "brier", "ece", "n", "positives"]]
if dataset == "dgraphfin":
    print("DGraph paper reference test AUC:", cfg.get("reference_auc"))
table.round(4)

## Is XGBoost really better than LR here?

Paired bootstrap over test rows: the difference in the metric between two models on the same resampled rows, 1000 times. Grey bars cross zero (no evidence); blue bars do not.

In [ ]:
from src.eval import bootstrap

compare = bootstrap.compare_many(y_te, preds["lr"][1], {k: v[1] for k, v in preds.items() if k != "lr"},
                                 metrics=("auc", "pr_auc", "capture_10"), n_boot=n_boot, seed=seed)
for r in compare:
    r["baseline"] = "lr"
cmp_df = pd.DataFrame(compare)
fig = viz.plot_intervals(cmp_df[cmp_df.metric == "auc"], title="Test AUC gain over LR (95% paired bootstrap)")
viz.savefig(fig, run_dir, "auc_vs_lr")
plt.show()
cmp_df[["model", "metric", "base", "new", "diff", "ci_low", "ci_high", "significant"]].round(4)

## Curves and calibration

In [ ]:
from src.eval.calibration import reliability_table

fig = viz.plot_roc_pr(y_te, {k: v[1] for k, v in preds.items()}, title=f"{dataset} test")
viz.savefig(fig, run_dir, "roc_pr")
plt.show()
fig = viz.plot_reliability({k: reliability_table(y_te, v[1]) for k, v in preds.items()}, title="(uncalibrated, test)")
viz.savefig(fig, run_dir, "reliability_raw")
plt.show()

## What drives the XGBoost score

Gain importance, top 25. These are the features any graph signal has to add to.

In [ ]:
imp = XGB.importance(booster, "gain")
fig = viz.plot_bar(imp.head(25), title="XGBoost gain importance (top 25)", ylabel="gain")
viz.savefig(fig, run_dir, "xgb_importance")
plt.show()
LR.coefficients(lr_model).head(15).to_frame("lr_coef")

In [ ]:
for name, (pv, pt) in preds.items():
    results.save_predictions(run_dir, name, {"val": (va, y_va, pv), "test": (te, y_te, pt)})
payload = {"models": rows, "compare": compare,
           "params": {"dataset": dataset, "seed": seed, "quick": quick, "models_cfg": models_cfg, "timings_s": timings,
                      "xgb": XGB.summary(booster), "mlp_epochs": len(history)}}
if dataset == "dgraphfin":
    payload["reference_auc"] = cfg.get("reference_auc")
results.save_metrics(run_dir, payload)
summary = {"test_auc": {m: round(r["test"]["auc"], 4) for m, r in rows.items()}, "timings_s": timings, "elapsed_s": round(time.time() - t0, 1)}
ckpt.mark_done(run_dir, summary)
summary